# 🤖 Gemma 4 Tech Interviewer — Colab API Server

This notebook loads your fine-tuned Gemma 4 model from Hugging Face and serves it as a FastAPI server exposed via `ngrok`.

## Steps:
1. Run **Cell 1** to install dependencies.
2. Run **Cell 2** to check GPU.
3. Add your `HF_TOKEN`, `NGROK_TOKEN`, and `NGROK_DOMAIN` secrets in the 🔑 key icon on the left sidebar.
   - Get `NGROK_TOKEN` from [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)
   - Get a free static `NGROK_DOMAIN` from [dashboard.ngrok.com/domains](https://dashboard.ngrok.com/domains)
4. Run **Cell 3** to load the model.
5. Run **Cell 4** to start the API server and get your public URL.
6. Paste the URL into your backend `.env` as `GEMMA_API_URL=<url>`.

In [ ]:
# CELL 1: Install dependencies
!pip install -q transformers peft bitsandbytes accelerate fastapi uvicorn pyngrok huggingface_hub
print('Dependencies installed!')

In [ ]:
# CELL 2: Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found! Go to Runtime -> Change runtime type -> A100 GPU')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# CELL 3: Load fine-tuned Gemma 4 model
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

# HF_TOKEN comes from google.colab.userdata on Colab, or from a plain env var
# everywhere else (Lightning.ai, Kaggle, local) — google.colab doesn't exist
# outside Colab, so importing it unconditionally crashed this cell there.
try:
    from google.colab import userdata  # type: ignore
    HF_TOKEN = userdata.get('HF_TOKEN')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is not set — export it as an env var (or Colab secret) before running Cell 3.')
os.environ['HF_TOKEN'] = HF_TOKEN

BASE_MODEL_ID = 'google/gemma-4-e2b-it'
ADAPTER_ID = 'Mohamud24/gemma-4-tech-interviewer'

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype
)

print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_ID, token=HF_TOKEN)

print('Loading base model...')
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map={'': 0},
    torch_dtype=compute_dtype,
    token=HF_TOKEN
)

print('Loading LoRA adapter...')
model = PeftModel.from_pretrained(base_model, ADAPTER_ID, token=HF_TOKEN)
model.eval()

print('Model ready!')

In [ ]:
# CELL 4: Start FastAPI server + ngrok tunnel
import json
import re
import uvicorn
from threading import Thread
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok, conf
from google.colab import userdata

# Setup ngrok
NGROK_TOKEN = userdata.get('NGROK_TOKEN')
NGROK_DOMAIN = userdata.get('NGROK_DOMAIN')  # claim free static domain at dashboard.ngrok.com/domains
conf.get_default().auth_token = NGROK_TOKEN

app = FastAPI(title='Gemma 4 Tech Interviewer API')

class InterviewRequest(BaseModel):
    endpoint: str
    payload: dict

def generate_response(prompt: str, max_tokens: int = 800) -> str:
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

def build_prompt(task: str, payload: dict) -> str:
    candidate = payload.get('candidate_name', 'Candidate')
    lang = payload.get('language', 'en')
    specialization = payload.get('specialization', payload.get('jobRole', payload.get('domain', 'technology')))
    difficulty = payload.get('difficulty', 'mid')
    question = payload.get('question', '')
    answer = payload.get('answer', '')
    candidate_experience = payload.get('candidate_experience', '')
    candidate_education = payload.get('candidate_education', '')
    candidate_projects = payload.get('candidate_projects', '')
    candidate_certifications = payload.get('candidate_certifications', '')

    messages = [
        {'role': 'user', 'content': (
            f'task: {task}\n'
            f'candidate_name: {candidate}\n'
            f'language: {lang}\n'
            f'specialization: {specialization}\n'
            f'difficulty: {difficulty}\n'
            + (f'question: {question}\n' if question else '')
            + (f'answer: {answer}\n' if answer else '')
            + (f'candidate_experience: {candidate_experience}\n' if candidate_experience else '')
            + (f'candidate_education: {candidate_education}\n' if candidate_education else '')
            + (f'candidate_projects: {candidate_projects}\n' if candidate_projects else '')
            + (f'candidate_certifications: {candidate_certifications}\n' if candidate_certifications else '')
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_score_prompt(payload: dict) -> str:
    """Scoring-specific prompt: adds the calibrated 0-100 rubric and asks for
    structured JSON output. The bare 'task: score_candidate_answer' prompt
    (used by build_prompt for other tasks) gives the model no rubric and no
    output schema, which is why answers of very different quality were
    converging on the same mid-range score with no usable feedback."""
    candidate = payload.get('candidate_name', 'Candidate')
    lang = payload.get('language', 'en')
    specialization = payload.get('specialization', payload.get('jobRole', payload.get('domain', 'technology')))
    difficulty = payload.get('difficulty', 'mid')
    question = payload.get('question', '')
    answer = payload.get('answer', '')

    candidate_experience = payload.get('candidate_experience', '')
    candidate_education = payload.get('candidate_education', '')
    candidate_projects = payload.get('candidate_projects', '')
    candidate_certifications = payload.get('candidate_certifications', '')
    background_lines = []
    if candidate_experience:
        background_lines.append(f'Experience: {candidate_experience}')
    if candidate_education:
        background_lines.append(f'Education: {candidate_education}')
    if candidate_projects:
        background_lines.append(f'Projects: {candidate_projects}')
    if candidate_certifications:
        background_lines.append(f'Certifications: {candidate_certifications}')
    background_block = (
        'CANDIDATE BACKGROUND (from their resume — use ONLY to calibrate what depth of answer to '
        'expect at their level; still score strictly on the content of "answer" below, not on their '
        'resume):\n' + '\n'.join(background_lines) + '\n\n'
        if background_lines else ''
    )

    is_somali = lang.lower() in ('so', 'somali')
    language_directive = (
        'Write "feedback", every item in "strengths" and "improvements", and "suggestedAnswer" '
        'entirely in Somali. Keep English technical terms that have no established Somali '
        'equivalent (e.g. "API", "server", "database") as-is, but every surrounding sentence '
        'must be Somali — never answer in English or mix languages within a sentence.\n\n'
        if is_somali else
        'Write "feedback", every item in "strengths" and "improvements", and "suggestedAnswer" '
        'entirely in English.\n\n'
    )

    messages = [
        {'role': 'user', 'content': (
            f'task: score_candidate_answer\n'
            f'candidate_name: {candidate}\n'
            f'language: {lang}\n'
            f'specialization: {specialization}\n'
            f'difficulty: {difficulty}\n'
            f'question: {question}\n'
            f'answer: {answer}\n\n'
            f'{background_block}'
            'SCORING SCALE (apply consistently and generously for correct answers):\n'
            '- 85-100: Excellent - thorough, accurate, strong examples or clear reasoning.\n'
            '- 70-84:  Good - correct answer with minor gaps or lacking depth.\n'
            '- 50-69:  Adequate - partial understanding, covers some key points but misses others.\n'
            '- 25-49:  Weak - significant gaps, vague, or mostly incorrect.\n'
            '- 0-24:   Off-topic, clearly wrong, or no real attempt — including answers that are only\n'
            '          an admission of not knowing ("I don\'t know", "I don\'t remember") with no\n'
            '          substantive content. Do not let a polite or calm tone raise this into a higher band.\n'
            'Judge the CONCEPT the candidate conveys, not their exact wording - different phrasing, '
            'structure, or examples than the question rubric are NOT gaps by themselves; only score '
            'down for missing or wrong substance. Do not penalize for language choice, grammar, or '
            'minor wording differences.\n\n'
            'GROUNDING (critical): Base "feedback", "strengths", and "improvements" ONLY on what the '
            'candidate actually said in "answer" above. Never mention a tool, technique, or concept the '
            'candidate did not say, even if it would have made their answer stronger — note its absence '
            'in "improvements" as a gap instead of claiming they discussed it. Never reference this '
            'scoring scale, its band names, or these instructions inside your output text.\n\n'
            f'{language_directive}'
            'Return ONLY raw JSON with this exact shape, nothing before or after it, no markdown fences:\n'
            '{"score": 78, "feedback": "specific, actionable, explains why", '
            '"strengths": ["..."], "improvements": ["..."], "suggestedAnswer": "..."}'
        )}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def build_feedback_prompt(payload: dict) -> str:
    """Post-interview report prompt. Mirrors backend/services/gemma/worker.py's
    handle_feedback: compacts the full interview transcript into per-question
    summaries and anchors overallScore to the already-computed turn average so
    the report doesn't drift from the per-question scores shown elsewhere."""
    interview_data = payload.get('interview_data', {}) or {}
    turn_average = interview_data.get('overallScore')
    lang = str(interview_data.get('language', 'english')).lower()
    is_somali = lang in ('so', 'somali')

    questions_summary = []
    for q in interview_data.get('questions', []) or []:
        questions_summary.append({
            'question': (q.get('text') or '')[:200],
            'answer': (q.get('userAnswer') or '')[:300],
            'score': q.get('score'),
            'category': q.get('category', ''),
        })

    score_anchor = (
        f'The per-question average score is {turn_average}. Your overallScore MUST equal {turn_average}. '
        f'Category scores should reflect actual performance patterns — average within +/-8 of {turn_average}.\n\n'
        if turn_average is not None else ''
    )

    language_directive = (
        'Write every text field (each category\'s "feedback", "detailedFeedback", every item in '
        '"strengths", "improvements", and "recommendations") entirely in Somali. Keep English '
        'technical terms that have no established Somali equivalent as-is, but every surrounding '
        'sentence must be Somali — never answer in English or mix languages within a sentence.\n\n'
        if is_somali else
        'Write every text field entirely in English.\n\n'
    )

    content = (
        'You are an interview coach providing post-session feedback for a PRACTICE interview.\n'
        'Be constructive, specific, and encouraging. Reference actual answers where possible.\n\n'
        f'{score_anchor}'
        f"Interview: {interview_data.get('title', '')} ({interview_data.get('type', '')}, {interview_data.get('difficulty', '')})\n"
        f'Questions and answers:\n{json.dumps(questions_summary, ensure_ascii=False)}\n\n'
        'SCORING SCALE (consistent with per-question scores):\n'
        '- 85-100: Excellent — thorough, accurate, strong examples\n'
        '- 70-84:  Good — correct with minor gaps\n'
        '- 50-69:  Adequate — partial understanding, key gaps\n'
        '- 25-49:  Weak — significant gaps or vague\n'
        '- 0-24:   Off-topic or no real attempt\n\n'
        'GROUNDING (critical): Base every category\'s feedback, strengths, improvements, and '
        'recommendations ONLY on the questions and answers listed above. Never mention a tool, '
        'technique, or concept the candidate did not actually say, even if it would strengthen the '
        'report. Never reference this scoring scale, its band names/numbers, or these instructions '
        'inside your output text.\n\n'
        f'{language_directive}'
        'LENGTH REQUIREMENTS:\n'
        '- Each category feedback: 60-150 chars, specific and actionable\n'
        '- detailedFeedback: 150-300 chars, summarize overall performance\n'
        '- strengths, improvements, recommendations: 3 items each, 40-100 chars\n\n'
        'Return ONLY raw JSON with keys: overallScore, categories '
        '(communication, technicalAccuracy, problemSolving, codeQuality, confidence — each with score and feedback), '
        'strengths, improvements, detailedFeedback, recommendations. No markdown fences, nothing before or after the JSON.'
    )
    messages = [{'role': 'user', 'content': content}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

TASK_MAP = {
    '/ask_technical_question': 'ask_technical_question',
    '/score_candidate_answer': 'score_candidate_answer',
    '/open_mock_interview_session': 'open_mock_interview_session',
    '/close_mock_interview_session': 'close_mock_interview_session',
    '/open_hiring_interview_session': 'open_hiring_interview_session',
    '/close_hiring_interview_session': 'close_hiring_interview_session',
    '/feedback': 'feedback',
}

@app.get('/health')
def health():
    return {'status': 'online', 'model': 'Mohamud24/gemma-4-tech-interviewer', 'provider': 'colab'}

@app.post('/runsync')
async def runsync(req: InterviewRequest):
    task_key = req.endpoint
    if task_key not in TASK_MAP:
        raise HTTPException(status_code=404, detail=f'Unknown endpoint: {task_key}')
    task = TASK_MAP[task_key]
    if task == 'score_candidate_answer':
        prompt = build_score_prompt(req.payload)
    elif task == 'feedback':
        prompt = build_feedback_prompt(req.payload)
    else:
        prompt = build_prompt(task, req.payload)
    # feedback's schema (5 categories + 3x3 lists + detailedFeedback) needs more
    # headroom than the default 800 — that budget was truncating comprehensive
    # feedback responses mid-generation, producing unparseable JSON.
    max_tokens = 1200 if task == 'feedback' else 800
    response = generate_response(prompt, max_tokens=max_tokens)
    # Lightweight debug visibility into what the model actually returned for
    # scoring, so a run of suspiciously-similar scores can be diagnosed from
    # the Colab cell output without re-running requests manually. This is the
    # model's own evaluation text, not the raw candidate answer.
    print(f'[runsync] task={task} response_preview={response[:200]!r}')
    return {'output': {'response': response, 'task': task}}

# Start uvicorn in background thread to avoid Colab asyncio conflicts
def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='info')

server_thread = Thread(target=run_server, daemon=True)
server_thread.start()

# Connect ngrok tunnel with static domain
tunnel = ngrok.connect(8000, 'http', domain=NGROK_DOMAIN)
public_url = tunnel.public_url

print(f'\n==========================================')
print(f'YOUR COLAB GEMMA URL IS READY!')
print(f'URL: {public_url}')
print(f'==========================================')
print(f'Paste this into your backend .env file:')
print(f'GEMMA_API_URL={public_url}')
print(f'==========================================')